#### TRANSFORM JOIN BETWEEN `GIZMO.SILVER.CUSTOMERS_DELTA` AND `GIZMO.SILVER.ADDRESSES_DELTA`

#### WRITE TRANSFORMED DATA TO GOLD SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: GOLD
3. TABLE NAME: CUSTOMERS_ORDERS_DELTA, ADDRESSES_ORDERS_DELTA

In [0]:
customers_delta_df = spark.table('''GIZMO.SILVER.CUSTOMERS_DELTA''')
display(customers_delta_df.limit(1))

In [0]:
addresses_delta_df = spark.table('''GIZMO.SILVER.ADDRESSES_DELTA''')
display(addresses_delta_df.limit(1))

#### JOIN BETWEEN `GIZMO.SILVER.CUSTOMERS_DELTA`, `GIZMO.SILVER.ADDRESSES_DELTA`
1. TABLES: `GIZMO.SILVER.CUSTOMERS_DELTA`, `GIZMO.SILVER.ADDRESSES_DELTA`
2. JOIN CONDITION = `customers_delta_df.customer_id == addresses_delta_df.customer_id`
3. JOIN TYPE: `INNER`

In [0]:
customers_addresses_df = (customers_delta_df.join(addresses_delta_df, customers_delta_df.customer_id == addresses_delta_df.customer_id, 'inner')
                          .select(customers_delta_df['customer_id'], customers_delta_df['first_name'], customers_delta_df['last_name'], 
                                  customers_delta_df['date_of_birth'], customers_delta_df['email'], customers_delta_df['telephone'],
                                  customers_delta_df['member_since'], addresses_delta_df['shipping_address_line_1'], addresses_delta_df['billing_address_line_1'],
                                  addresses_delta_df['billing_city'], addresses_delta_df['billing_state'], addresses_delta_df['shipping_postcode'],
                                  addresses_delta_df['billing_postcode']))

display(customers_addresses_df.limit(1))
print(f'Rows Effected: {customers_addresses_df.count()}')

In [0]:
customers_addresses_df.writeTo('GIZMO.GOLD.CUSTOMERS_ORDERS_DELTA').createOrReplace()

In [0]:
cust_address_summary_count_df = spark.sql('''SELECT * FROM GIZMO.GOLD.CUSTOMERS_ORDERS_DELTA''');
print(f'Row Count: {cust_address_summary_count_df.count()}')

#### QUERY AND VALIDATE THE RECORDS

In [0]:
%run /Workspace/Users/pde1409@hotmail.com/AzureDatabricks-Gizmo/AzureDatabricks-GizmoBox/01.GizmoBox/02.Config/01.config

In [0]:
try:
    verify_pipeline_counts(customers_addresses_df, cust_address_summary_count_df, "PySpark-07.TransformCustomersAddresses")
except AssertionError as e:
    raise

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime
import uuid
import sys
import traceback
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

# =====================================================================
# 1. INITIALIZE PIPELINE METADATA
# =====================================================================
# Capture start time at the absolute beginning of the execution
load_start_time = datetime.now()
pipeline_name = 'PySpark-07.TransformCustomersAddresses'

# Establish default tracking states
status = "SUCCESS"
message = "Transformed and loaded Customer Address data into Gold Table"
record_count = 0

try:
    # =====================================================================
    # 2. CORE ETL LOGIC
    # =====================================================================
    
    # Step A: Extract Data using absolute path and explicit format configuration
    customers_addresses_df = (customers_delta_df.join(addresses_delta_df, customers_delta_df.customer_id == addresses_delta_df.customer_id, 'inner')
                          .select(customers_delta_df['customer_id'], customers_delta_df['first_name'], customers_delta_df['last_name'], 
                                  customers_delta_df['date_of_birth'], customers_delta_df['email'], customers_delta_df['telephone'],
                                  customers_delta_df['member_since'], addresses_delta_df['shipping_address_line_1'], addresses_delta_df['billing_address_line_1'],
                                  addresses_delta_df['billing_city'], addresses_delta_df['billing_state'], addresses_delta_df['shipping_postcode'],
                                  addresses_delta_df['billing_postcode'])) 
    
    # Step B: Execute target transformations or loading actions here
    # (Example: customers_addresses_df.write.mode("overwrite").saveAsTable("GIZMO.GOLD.CUSTOMERS_ADDRESSES"))
    
    # Step C: Capture final evaluated source record count
    record_count = customers_addresses_df.count()
    
    # =====================================================================

except Exception as e:
    # 3. EXCEPTION HANDLING
    # If any error occurs above, catch it, flip status, and parse the trace
    status = "FAILED"
    
    exc_type, exc_value, exc_tb = sys.exc_info()
    error_details = traceback.format_exception_only(exc_type, exc_value)[0].strip()
    message = f"Pipeline failed! Error: {error_details}"
    record_count = -1  # Standard indicator flag representing an uncompleted execution

finally:
    # =====================================================================
    # 4. AUDIT & LOGGING (Guaranteed execution via finally block)
    # =====================================================================
    load_end_time = datetime.now()
    current_date = load_end_time.date()

    # Step A: Calculate Sequential Run ID for Today (Scoped to THIS specific pipeline)
    try:
        # Rectified: Convert Python date object to string to guarantee reliable evaluation in Spark
        current_date_str = current_date.strftime('%Y-%m-%d')
        
        max_run_df = spark.table("GIZMO.AUDIT.AUDIT_LOGS") \
            .filter(
                (F.col("event_time") == F.lit(current_date_str)) & 
                (F.col("pipeline_name") == F.lit(pipeline_name))
            ) \
            .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
        
        max_id_row = max_run_df.collect()[0]
        next_run_int = (max_id_row["max_id"] + 1) if max_id_row["max_id"] is not None else 1
    except Exception:
        # Defaults to 1 if table is empty, uninitialized, or completely drops out
        next_run_int = 1

    # Apply 2-digit zero padding format string (e.g., 1 -> "01", 11 -> "11")
    run_id_str = f"{next_run_int:02d}"

    # Step B: Secure Notebook Cluster Context Metadata safely
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
        user_name = context.tags().apply("user")
    except Exception:
        notebook_path = "Unknown/Local"
        user_name = "System"

    # Step C: Package the metadata tracking Row
    log_entry = Row(
        log_id=str(uuid.uuid4()),
        run_id=run_id_str,                
        event_time=current_date,          
        event_type="FULL LOAD",
        source_table="GIZMO.SILVER.CUSTOMERS_DELTA, GIZMO.SILVER.ADDRESSES_DELTA",
        target_table="GIZMO.GOLD.CUSTOMERS_ADDRESSES",
        record_count=record_count,
        status=status,                    
        message=message,                   
        user_name=user_name,
        notebook_path=notebook_path,
        pipeline_name=pipeline_name,
        load_start_time=load_start_time,  
        load_end_time=load_end_time       
    )

    # Step D: Declare structured explicit Schema types matching Target DDL exactly
    log_schema = StructType([
        StructField("log_id", StringType(), True),
        StructField("run_id", StringType(), True),  
        StructField("event_time", DateType(), True),
        StructField("event_type", StringType(), True),
        StructField("source_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("notebook_path", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("load_start_time", TimestampType(), True),
        StructField("load_end_time", TimestampType(), True)
    ])

    # Step E: Instantiate log DataFrame and append transactional trace record
    log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
    log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.AUDIT.AUDIT_LOGS")
    print(f"[AUDIT LOGGED] Status: {status} | Run ID: {run_id_str} | Count: {record_count}")

    # Step F: Force a hard stop exception for workflow orchestrators if pipeline failed
    if status == "FAILED":
        raise RuntimeError(message)

In [0]:
%sql
SELECT 
  run_id, 
  event_time, 
  pipeline_name, 
  record_count,
  date_format(FROM_UTC_TIMESTAMP(load_start_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_start_time_ist, 
  date_format(FROM_UTC_TIMESTAMP(load_end_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_end_time_ist
FROM GIZMO.AUDIT.AUDIT_LOGS
WHERE pipeline_name = 'PySpark-07.TransformCustomersAddresses'
ORDER BY pipeline_name, event_time DESC, run_id ASC;

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS ADDRESSES LOADED INTO GIZMO.GOLD.CUSTOMERS_ADDRESS_DELTA")